# Mise en production, indexation avec l'API d'Anthropic

Projet TorneH, fonds Éric Jantzen du musée d'Orsay. Ce notebook réalise la même tâche que molmo_batch, une détection à vocabulaire contrôlé, avec un modèle hébergé, Claude Sonnet 5, appelé par l'API Batches. Il a été lancé après un test sur 10 images sur 3000 photographies dites de référence, ce qui permet de comparer une solution payante à une solution ouverte limitée par le GPU.

- Entrée, un dossier d'images et le thésaurus json lu par ses champs Term_TMS et TermID, le prompt et les termes étant en français
- Traitement, redimensionnement des images à 1280 pixels, envoi par batchs de 400, prompt du thésaurus identique pour toutes les requêtes et mis en cache, puis récupération asynchrone des résultats
- Sortie, un fichier resultats.json avec, par image, le nom du fichier, les termes reconnus avec leur TermID, les termes hors liste et l'éventuelle erreur
- Suivi, un fichier batches.json conserve les identifiants des batchs, ce qui permet de fermer Colab et de reprendre avec status puis collect
- Fonctions principales, load_terms, encode, copy_images, submit, status et collect

Le fichier thesau_angl.json et le dossier images sont à placer dans index_jantzen/claude_API du Google Drive. La clé API se déclare dans les Secrets de Colab sous le nom ANTHROPIC_API_KEY.

In [ ]:
# Installation du client de l'API et de pillow, utilisé pour redimensionner les images.
!pip install -q anthropic pillow

# Montage du Google Drive et lecture de la clé API depuis les secrets de Colab, afin qu'elle n'apparaisse pas dans le notebook.
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Indexation des images du fonds avec Claude Sonnet 5 par l'API Batches.
# Le thésaurus utilisé est la version française, avec les termes Term_TMS, et le prompt est en français.
import base64, io, json, shutil
from pathlib import Path
import anthropic
from PIL import Image

# Configuration. Les chemins pointent vers le dossier de travail du Drive. Le fichier STATE conserve les identifiants des batchs, ce qui permet de reprendre le suivi après fermeture de Colab. Le fichier OUT reçoit les résultats.
WORK = Path("/content/drive/MyDrive/index_jantzen/claude_API")
THESAURUS = WORK / "thesau_angl.json"
IMG_SRC = WORK / "images"
IMG_DIR = Path("/content/images")      # copie locale, lecture plus rapide
STATE = WORK / "batches.json"
OUT = WORK / "resultats.json"

# Paramètres de l'API. Les images sont envoyées par batchs de 400 en raison de la limite de taille par batch, et réduites à 1280 pixels de côté pour limiter le volume de données.
MODEL = "claude-sonnet-5"
CHUNK = 400          # images par batch, limite de 256 Mo par batch
MAX_SIDE = 1280
EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".webp"}

# Prompt de détection, équivalent à celui des autres notebooks. La liste des termes est insérée à la place de terms.
PROMPT = """Voici une liste contrôlée de mots-clés.

{terms}

Analyse attentivement l'image fournie et identifie UNIQUEMENT les éléments architecturaux de cette liste qui sont réellement visibles sur l'image.

Contraintes strictes.
N'utilise que des mots présents dans la liste, recopiés à l'identique.
N'invente aucun mot hors de la liste.
Ne donne aucune explication, aucune phrase.
Réponds uniquement sous forme d'une liste séparée par des virgules."""

# Client de l'API, la clé étant lue dans la variable d'environnement ANTHROPIC_API_KEY.
client = anthropic.Anthropic()

def load_terms():
    """Lit le thésaurus et renvoie un dictionnaire associant chaque Term_TMS à son TermID. Le parcours est récursif pour s'adapter à une structure imbriquée du json."""
    data = json.loads(THESAURUS.read_text(encoding="utf-8"))
    terms = {}
    def walk(x):
        """Parcourt récursivement le json et relève les termes."""
        if isinstance(x, dict):
            if "Term_TMS" in x and "TermID" in x:
                terms.setdefault(str(x["Term_TMS"]).strip(), x["TermID"])
            for v in x.values():
                walk(v)
        elif isinstance(x, list):
            for i in x:
                walk(i)
    walk(data)
    return terms

def encode(path):
    """Redimensionne une image et la renvoie encodée en base64 au format JPEG, pour l'envoi à l'API."""
    im = Image.open(path).convert("RGB")
    im.thumbnail((MAX_SIDE, MAX_SIDE))
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()

def copy_images():
    """Copie les images du Drive vers le disque local de Colab, où la lecture est plus rapide. Les images déjà copiées sont ignorées."""
    IMG_DIR.mkdir(exist_ok=True)
    for p in IMG_SRC.iterdir():
        if p.suffix.lower() in EXTS and not (IMG_DIR / p.name).exists():
            shutil.copy(p, IMG_DIR / p.name)
    print(len(list(IMG_DIR.iterdir())), "images en local")

def submit(limit=None):
    """Envoie les images à l'API par batchs. Le paramètre limit restreint le nombre d'images, pour les tests. Le prompt contenant le thésaurus est identique pour toutes les requêtes et mis en cache, ce qui réduit le coût. Les identifiants des batchs et la correspondance entre identifiant de requête et nom de fichier sont sauvegardés dans STATE."""
    copy_images()
    files = sorted(p for p in IMG_DIR.iterdir() if p.suffix.lower() in EXTS)
    if limit:
        files = files[:limit]
    terms = sorted(load_terms())
    print(len(terms), "termes dans le thésaurus")
    system = [{"type": "text", "text": PROMPT.format(terms="\n".join(terms)),
               "cache_control": {"type": "ephemeral"}}]
    names, batch_ids = {}, []
    for i in range(0, len(files), CHUNK):
        reqs = []
        for j, p in enumerate(files[i:i + CHUNK], start=i):
            cid = f"img{j:05d}"
            names[cid] = p.name
            reqs.append({
                "custom_id": cid,
                  "params": {
                    "model": MODEL,
                    "max_tokens": 500,

                    # Désactivation du mode de réflexion, la tâche demandant seulement une liste de termes.
                    "thinking": {"type": "disabled"},
                    "system": system,
                    "messages": [{"role": "user", "content": [
                        {"type": "image", "source": {"type": "base64",
                         "media_type": "image/jpeg", "data": encode(p)}},
                        {"type": "text", "text": "Image à analyser."}]}],
                },
            })
        b = client.messages.batches.create(requests=reqs)
        batch_ids.append(b.id)
        print("batch", b.id, len(reqs), "images")
    STATE.write_text(json.dumps({"batches": batch_ids, "names": names}, indent=1))

def status():
    """Affiche l'état d'avancement de chaque batch."""
    state = json.loads(STATE.read_text())
    for bid in state["batches"]:
        b = client.messages.batches.retrieve(bid)
        print(bid, b.processing_status, b.request_counts)

def collect():
    """Récupère les résultats des batchs terminés et les enregistre dans OUT. Chaque terme renvoyé est vérifié par rapport au thésaurus. Les termes reconnus sont conservés avec leur TermID et les termes hors liste sont listés à part, pour mesurer les inventions du modèle. Les requêtes en échec sont consignées avec leur type d'erreur."""
    state = json.loads(STATE.read_text())
    valid = load_terms()
    out = []
    for bid in state["batches"]:
        b = client.messages.batches.retrieve(bid)
        if b.processing_status != "ended":
            print(bid, "en cours", b.request_counts)
            continue
        for r in client.messages.batches.results(bid):
            name = state["names"][r.custom_id]
            if r.result.type != "succeeded":
                out.append({"fichier": name, "termes": [], "hors_liste": [], "erreur": r.result.type})
                continue
            text = r.result.message.content[0].text.strip()
            found = list(dict.fromkeys(t.strip() for t in text.split(",") if t.strip()))
            out.append({
                "fichier": name,
                "termes": [{"Term_TMS": t, "TermID": valid[t]} for t in found if t in valid],
                "hors_liste": [t for t in found if t not in valid],
                "erreur": "",
            })
    out.sort(key=lambda d: d["fichier"])
    OUT.write_text(json.dumps(out, ensure_ascii=False, indent=1), encoding="utf-8")
    print(len(out), "images écrites dans", OUT)
    return out

## 1. Test sur 10 images

In [ ]:
# Test sur 10 images avant le traitement complet.
submit(10)

3067 images en local
96 termes dans le thésaurus
batch msgbatch_01C2jYwPuDkk6buRDsvCGMRW 10 images


## 2. Suivi des batchs

In [ ]:
# Vérification de l'état des batchs.
status()

msgbatch_014U2nmksM5AcM1fYZ9GpxeF ended MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=400)
msgbatch_01Hc698ynyDJrbZCf7eg4CZL ended MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=400)
msgbatch_018P3T92sGt6rmaksqNibbZ6 ended MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=400)
msgbatch_01KRhGtTDZSYVxC55DG83U6e ended MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=400)
msgbatch_01FbUgWD2bEhu8LHd249xPwa ended MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=400)
msgbatch_013NgDn7LuEWsNxEWwPpik2m ended MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=400)
msgbatch_01Nt4XM74V8dALE2vC4nt54w ended MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=400)
msgbatch_01883HMeixVCPN3CGkK9CAab ended MessageBatchRequestCounts(canceled=0, errored=0, expired=

## 3. Récupération des résultats

In [ ]:
# Récupération des résultats et affichage des trois premiers pour contrôler le format.
res = collect()
res[:3]

3067 images écrites dans /content/drive/MyDrive/index_jantzen/claude_API/resultats.json


[{'fichier': 'Abbaye-2_6__1.jpg',
  'termes': [{'Term_TMS': 'fenêtre', 'TermID': '224-234'},
   {'Term_TMS': 'mansarde', 'TermID': '224-312'},
   {'Term_TMS': 'lucarne', 'TermID': '224-310'},
   {'Term_TMS': 'garde-corps', 'TermID': '224-282'},
   {'Term_TMS': 'ardoise', 'TermID': '223 817'},
   {'Term_TMS': 'souche de cheminée', 'TermID': '224-315'},
   {'Term_TMS': 'porte', 'TermID': '224-246'},
   {'Term_TMS': 'tuile', 'TermID': '223 859'},
   {'Term_TMS': 'balcon', 'TermID': '224-495'},
   {'Term_TMS': 'pierre de taille', 'TermID': '223 844'},
   {'Term_TMS': 'volet', 'TermID': '224-296'},
   {'Term_TMS': 'fer forgé', 'TermID': '223 832'},
   {'Term_TMS': 'travée', 'TermID': '224-561'}],
  'hors_liste': ['vitrine'],
  'erreur': ''},
 {'fichier': 'Abbaye-3_6__1-Rx.jpg',
  'termes': [{'Term_TMS': 'lucarne', 'TermID': '224-310'},
   {'Term_TMS': 'fronton', 'TermID': '224-396'},
   {'Term_TMS': 'arc en plein cintre', 'TermID': '224-330'},
   {'Term_TMS': 'fenêtre', 'TermID': '224-234'}

## 4. Traitement complet

Le test est à vérifier avant de lancer cette cellule, qui écrase batches.json. Le traitement s'effectuant chez Anthropic, Colab peut être fermé puis rouvert pour lancer status et collect.

In [ ]:
# Traitement de l'ensemble du fonds.
submit()

3067 images en local
96 termes dans le thésaurus
batch msgbatch_014U2nmksM5AcM1fYZ9GpxeF 400 images
batch msgbatch_01Hc698ynyDJrbZCf7eg4CZL 400 images
batch msgbatch_018P3T92sGt6rmaksqNibbZ6 400 images
batch msgbatch_01KRhGtTDZSYVxC55DG83U6e 400 images
batch msgbatch_01FbUgWD2bEhu8LHd249xPwa 400 images
batch msgbatch_013NgDn7LuEWsNxEWwPpik2m 400 images
batch msgbatch_01Nt4XM74V8dALE2vC4nt54w 400 images
batch msgbatch_01883HMeixVCPN3CGkK9CAab 267 images
